# Session 7 · Case study: honest validation on reviews and churn

*Author: course team · Licence: CC-BY-4.0*

This workbook contains the practice tasks of all three blocks of Session 7:

| Part | Block | Task |
|---|---|---|
| A | 0:00–0:45 | Cross-validate the regression and logistic models of Session 6 and report the scores with confidence intervals |
| B | 1:00–1:45 | Tune the ridge penalty and the regularisation of logistic regression; compare random and time-based validation on the reviews |
| C | 2:00–2:45 | Demonstrate a leaking workflow, then fix it with a Pipeline and compare the scores |

**Data.** The review sample `case-study/data/train_sample.parquet` (50,000 reviews, prepared with `case-study/prepare_data.py`) and the IBM Telco churn data (downloaded from GitHub).

Tasks for you are marked **Task**. Each part ends with a short question to answer in a Markdown cell.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import f1_score, r2_score, roc_auc_score
from sklearn.model_selection import (GridSearchCV, KFold, StratifiedKFold, TimeSeriesSplit,
                                     cross_val_score, train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# find the repository root (the folder that contains case-study/)
ROOT = Path.cwd()
while not (ROOT / "case-study").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "case-study" / "data"

rev = pd.read_parquet(DATA / "train_sample.parquet").sort_values("date", ignore_index=True)
print(rev.shape, rev["date"].min().date(), rev["date"].max().date())

The seven simple text features of Session 2. They are computed row by row with fixed rules (no fitting), so it is safe to compute them before splitting.

In [ ]:
def simple_features(d):
    t = d["title"].fillna("") + " " + d["text"].fillna("")
    return pd.DataFrame({
        "log_len": np.log1p(t.str.len()), "n_excl": t.str.count("!"), "n_quest": t.str.count(r"\?"),
        "n_neg": t.str.lower().str.count(r"\b(?:not|no|never|don't|didn't|doesn't|waste|return)\b"),
        "verified": d["verified_purchase"].astype(int), "log_helpful": np.log1p(d["helpful_vote"]),
        "n_images": d["n_images"]})

Xs = simple_features(rev)
y_label = rev["label"]
Xs.describe().round(2)

A helper for the bootstrap confidence interval of any metric on a fixed set of predictions (theory page 01).

In [ ]:
def bootstrap_ci(y_true, y_pred, metric, n_boot=1000, seed=0):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    rng = np.random.default_rng(seed)
    stats = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y_true), len(y_true))
        stats.append(metric(y_true[i], y_pred[i]))
    return np.percentile(stats, [2.5, 97.5])

## Part A · Cross-validated scores with confidence intervals

### A1 · Regression of helpful votes (Session 6)

Target: `log1p(helpful_vote)`. Features: text length, exclamation and question marks, negation words, verified purchase, number of images and the star rating. We report

1. the 5-fold cross-validated R² (mean and standard deviation across folds), and
2. the R² on a held-out test set (newest 20 % of reviews) with a bootstrap 95 % interval.

In [ ]:
X_reg = Xs.drop(columns="log_helpful").assign(rating=rev["rating"])
y_reg = Xs["log_helpful"]

lin = make_pipeline(StandardScaler(), LinearRegression())
cv5 = KFold(5, shuffle=True, random_state=0)
r2 = cross_val_score(lin, X_reg, y_reg, cv=cv5, scoring="r2")
print("R² per fold:", r2.round(3), " mean", r2.mean().round(3), " sd", r2.std().round(3))

cut = int(0.8 * len(rev))                              # rows are sorted by date
lin.fit(X_reg.iloc[:cut], y_reg.iloc[:cut])
pred = lin.predict(X_reg.iloc[cut:])
print("test R²:", round(r2_score(y_reg.iloc[cut:], pred), 3),
      " 95% CI:", bootstrap_ci(y_reg.iloc[cut:], pred, r2_score).round(3))

Look closely: the random 5-fold CV gives a clearly positive R², but on the newest 20 % of reviews (from October 2020) the R² is close to zero or negative. Helpful votes accumulate over time, so recent reviews have had little time to collect votes: the relation between features and target has shifted. A random split hides this; an out-of-time test shows it.

### A2 · Logistic regression for Telco churn (Session 6)

We report cross-validated ROC AUC and a bootstrap interval on a stratified 25 % test set.

In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
telco = pd.read_csv(URL)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce").fillna(0)
y_churn = (telco["Churn"] == "Yes").astype(int)
X_churn = telco.drop(columns=["customerID", "Churn"])
num = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
cat = [c for c in X_churn.columns if c not in num]
churn_model = make_pipeline(
    make_column_transformer((StandardScaler(), num), (OneHotEncoder(handle_unknown="ignore"), cat)),
    LogisticRegression(max_iter=1000))

skf = StratifiedKFold(5, shuffle=True, random_state=0)
auc = cross_val_score(churn_model, X_churn, y_churn, cv=skf, scoring="roc_auc")
print("AUC per fold:", auc.round(3), " mean", auc.mean().round(3), " sd", auc.std().round(3))

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_churn, y_churn, test_size=0.25, stratify=y_churn, random_state=0)
proba = churn_model.fit(Xc_tr, yc_tr).predict_proba(Xc_te)[:, 1]
print("test AUC:", round(roc_auc_score(yc_te, proba), 3),
      " 95% CI:", bootstrap_ci(yc_te, proba, roc_auc_score).round(3))

### A3 · The review classifier (leaderboard round L1 model)

Macro-F1 of logistic regression on the seven features, 5-fold stratified CV and an out-of-time test (newest 20 %).

In [ ]:
l1 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
f1s = cross_val_score(l1, Xs, y_label, cv=skf, scoring="f1_macro")
print("macro-F1 per fold:", f1s.round(3), " mean", f1s.mean().round(3))

l1.fit(Xs.iloc[:cut], y_label.iloc[:cut])
pred_lab = l1.predict(Xs.iloc[cut:])
macro_f1 = lambda a, b: f1_score(a, b, average="macro")
print("out-of-time macro-F1:", round(macro_f1(y_label.iloc[cut:], pred_lab), 3),
      " 95% CI:", bootstrap_ci(y_label.iloc[cut:], pred_lab, macro_f1).round(3))

**Task A.** Collect the three results in one table (model, metric, CV mean ± sd, test score, 95 % interval). Is the test score inside the range you would expect from cross-validation? Which of the three estimates is most uncertain, and why?

In [ ]:
# your table here, for example:
# pd.DataFrame([{"model": "linear regression", "metric": "R²", "cv_mean": ..., "cv_sd": ..., "test": ..., "ci_low": ..., "ci_high": ...}, ...])

## Part B · Tuning and the choice of validation scheme

### B1 · The ridge penalty for helpful votes

With seven features ridge regression cannot do much: there is little to shrink. We add all pairwise products (`PolynomialFeatures(degree=2)`), which gives the penalty something to control, and search `alpha` on a log grid.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

ridge = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), StandardScaler(), Ridge())
grid = GridSearchCV(ridge, {"ridge__alpha": np.logspace(-2, 5, 8)}, cv=cv5, scoring="r2", return_train_score=True)
grid.fit(X_reg.iloc[:cut], y_reg.iloc[:cut])
res = pd.DataFrame(grid.cv_results_)[["param_ridge__alpha", "mean_train_score", "mean_test_score", "std_test_score"]]
print(res.round(4).to_string(index=False))
print("best:", grid.best_params_)

### B2 · The regularisation `C` of the review classifier: random versus time-based validation

The leaderboard trains on the past and tests on the future. Does a random `StratifiedKFold` give the same picture as a `TimeSeriesSplit`? We tune `C` with both and compare the chosen value and the scores.

In [ ]:
param = {"logisticregression__C": [0.001, 0.01, 0.1, 1, 10]}
dev_X, dev_y = Xs.iloc[:cut], y_label.iloc[:cut]          # development data: oldest 80 %
schemes = {"random (StratifiedKFold)": StratifiedKFold(5, shuffle=True, random_state=0),
           "time (TimeSeriesSplit)": TimeSeriesSplit(5)}
rows = []
for name, splitter in schemes.items():
    gs = GridSearchCV(l1, param, cv=splitter, scoring="f1_macro").fit(dev_X, dev_y)
    test_f1 = macro_f1(y_label.iloc[cut:], gs.predict(Xs.iloc[cut:]))
    rows.append({"scheme": name, "best_C": gs.best_params_["logisticregression__C"],
                 "cv_macro_f1": round(gs.best_score_, 3), "test_macro_f1": round(test_f1, 3)})
pd.DataFrame(rows)

In [ ]:
# per-fold scores of the time-based split: does performance change over the years?
tss = TimeSeriesSplit(5)
for tr, va in tss.split(dev_X):
    l1.fit(dev_X.iloc[tr], dev_y.iloc[tr])
    s = macro_f1(dev_y.iloc[va], l1.predict(dev_X.iloc[va]))
    print(rev["date"].iloc[va].min().date(), "to", rev["date"].iloc[va].max().date(), " macro-F1", round(s, 3),
          " share neg", round((dev_y.iloc[va] == "neg").mean(), 3))

**Task B.** (1) Which `alpha` and which `C` were chosen? Is the optimum flat or sharp? If the best value lies at the edge of the grid, extend the grid (for example `C` down to 1e-5) and search again. (2) Do random and time-based validation agree here? Look at the share of negative reviews per period: what would you expect for features that depend more on time (for example product averages or new words)? (3) Which scheme would you use for the leaderboard and why?

## Part C · A leaking workflow and its fix

### C1 · Preprocessing leakage: word selection outside the cross-validation

On 1,000 reviews we count words and keep the 100 words most associated with the label (chi-squared test). The **leaking** version selects the words on all 1,000 reviews and then cross-validates; the **correct** version puts vectoriser, selector and model into one pipeline.

In [ ]:
small = rev.sample(1000, random_state=0)
text = small["title"].fillna("") + " " + small["text"].fillna("")
clf = LogisticRegression(max_iter=2000, class_weight="balanced")

# leaking: vocabulary and word selection see all rows, including the later validation folds
counts = CountVectorizer(min_df=2).fit_transform(text)
X_leak = SelectKBest(chi2, k=100).fit_transform(counts, small["label"])
leak = cross_val_score(clf, X_leak, small["label"], cv=skf, scoring="f1_macro")

# correct: every step is refitted on the training folds only
pipe = make_pipeline(CountVectorizer(min_df=2), SelectKBest(chi2, k=100), clf)
clean = cross_val_score(pipe, text, small["label"], cv=skf, scoring="f1_macro")
print("leaking :", leak.round(3), leak.mean().round(3))
print("pipeline:", clean.round(3), clean.mean().round(3))

Which score is closer to the truth? Check on reviews that neither workflow has seen: the next 5,000 reviews in the sample.

In [ ]:
fresh = rev.drop(small.index).sample(5000, random_state=1)
fresh_text = fresh["title"].fillna("") + " " + fresh["text"].fillna("")
pipe.fit(text, small["label"])
print("macro-F1 on 5,000 fresh reviews:", round(macro_f1(fresh["label"], pipe.predict(fresh_text)), 3))

### C2 · Target leakage: the product's average rating

`train_avg_rating` in the product table is the mean rating of the product over the whole training period, so for a training review it *includes the review's own rating*. It will not be available in that form for future reviews.

In [ ]:
prod = pd.read_parquet(DATA / "products.parquet", columns=["parent_asin", "train_avg_rating", "train_n_reviews"])
d = rev.merge(prod, on="parent_asin", how="left")
X_target_leak = Xs.assign(avg_rating=d["train_avg_rating"])
for name, X_ in [("7 features", Xs), ("7 features + train_avg_rating", X_target_leak)]:
    s = cross_val_score(l1, X_, y_label, cv=TimeSeriesSplit(5), scoring="f1_macro")
    print(f"{name:32s} time-based CV macro-F1 {s.mean():.3f}")
print("share of reviews whose product has a single training review:", round((d["train_n_reviews"] == 1).mean(), 3))

For products with a single training review, `train_avg_rating` *is* the rating, and therefore reveals the label. In a trial leaderboard run (see `case-study/README.md`) this feature gave 0.55 in time-based CV but only 0.41 on the 2022 test reviews. Session 9 builds a leak-free version (the mean of *earlier* reviews of the same product).

### C3 · Summary table

In [ ]:
summary = pd.DataFrame([
    {"workflow": "word selection outside CV (leaking)", "cv_macro_f1": leak.mean()},
    {"workflow": "word selection inside a Pipeline", "cv_macro_f1": clean.mean()},
])
summary.round(3)

**Task C.** (1) Add the two rows of C2 to the summary table. (2) Explain in two sentences why the leaking workflow scores higher, and which number you would report. (3) Look at the columns of the review and product tables. Find one more feature that could leak information about the label, and explain how you would compute it without leakage.

**Team project.** Write a half-page validation plan for your project model: which splitter (and why), which metric, how you will report uncertainty, and how you will avoid leakage.